# Final variable appending and organization

Now, all needed variables (chl, cdom, and rrs) have been organized and standardized. For the final dataset, the chlorophyll dataset will be matched to the cdom and rrs and any coincidental matchups appended. This final dataset is therefore the same length as the chlorophyll one, but has more columns.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import LONGITUDE_FORMATTER, LATITUDE_FORMATTER
import geopandas as gpd
from datetime import datetime
import os
from matplotlib import ticker
import datetime as dt
import plotly.express as px
import cmocean as cm
import cmocean.cm as cmo
import matplotlib.gridspec as gridspec
import time
import matplotlib.ticker as mticker

In [2]:
#load in all completed dataframes
chl = pd.read_excel(r'C:\Users\gianna.milton\Documents\Python\Coastal_chl_final\all_chl.xlsx')
cdom = pd.read_excel(r'C:\Users\gianna.milton\Documents\Python\Coastal_chl_final\all_cdom.xlsx')

In [ ]:
rrs = pd.read_excel(r'C:\Users\gianna.milton\Documents\Python\Coastal_chl_final\all_rrs.xlsx')
rrs = rrs[rrs['wavelength'] <=800].reset_index(drop=True) #reduce to more realistic limit of wavelengths

In [ ]:
#first, turn rrs data long format so that there is 1 uinique id for each row / group of wavelengths
def long_to_wide(df):
    """
    Transforms long format to wide format (rrs_###) without averaging duplicates,
    preserving rows even if they have missing metadata (NaNs).
    """
    id_vars = ['source', 'datetime', 'lon', 'lat', 'depth', 'experiment', 'DOI_url','affiliations', 'investigators', 'contact', 'cruise', 'station',
               'ID_code']
    df_temp = df.copy()
    # temporarily fill NaNs with a string so pivot_table doesn't drop them
    df_temp[id_vars] = df_temp[id_vars].fillna('MISSING_DATA')
    
    #group by the full id_vars list to ensure the counter perfectly aligns
    df_temp['temp_counter'] = df_temp.groupby(id_vars + ['wavelength']).cumcount()
    df_wide = df_temp.pivot_table(index=id_vars + ['temp_counter'], columns='wavelength',  values='rrs')
    new_column_names = [f"rrs_{str(col)}" for col in df_wide.columns]
    df_wide.columns = new_column_names
    df_wide = df_wide.reset_index()
    df_wide = df_wide.drop(columns=['temp_counter'])
    return df_wide

rrs_long = long_to_wide(rrs)
rrs_long=rrs_long.replace('MISSING_DATA', np.nan)

For appending these variables, first they are reduced to just the variable and ID_column

In [ ]:
cdom_reduced = cdom[['cdom', 'ID_code']]
rrs_cols = [col for col in rrs_long.columns if col.startswith('rrs_')]
rrs_reduced = rrs_long[rrs_cols+['ID_code']]

Next, the are merged with chl based on the ID_code. 

In [ ]:
alldata_chl = pd.merge(chl, cdom_reduced, on=['ID_code'], how='outer').reset_index(drop=True)
alldata_chl = alldata_chl.dropna(subset=['chl', 'chl_a'], how='all')
#filtered_df = df_merged[df_merged['chl'].notna() & df_merged['cdom'].notna()] 

In [ ]:
alldata_chl = pd.merge(alldata_chl, rrs_reduced, on=['ID_code'], how='outer').reset_index(drop=True)
alldata_chl = alldata_chl.dropna(subset=['chl', 'chl_a'], how='all')

In [ ]:
#alldata_chl.to_excel('compiled_chl_variables.xlsx', index = False) #all chlorophyll data with coincidental rrs and cdom data
filename = "compiled_chl_variables.csv.gz"
alldata_chl.to_csv(filename, index=False, compression='gzip')